# 23장 · 문서와 출처를 반환하는 Retriever 만들기

22장에서 library-1이 가장 가깝다는 것을 확인했습니다. 이제 답을 쓰는 쪽에 검색 결과를 건네야 합니다. 저장소의 열별 목록을 ‘한 건의 ID, 본문, 출처, 쪽’이 붙은 문서 목록으로 정리합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 첫 질문의 첫 번째 결과 읽기

Chroma의 질의 응답은 여러 질문을 한 번에 받을 수 있게 질문별로 다시 묶여 있습니다. 아래 자료는 실제 서버를 호출하지 않고 그 모양만 담아 둔 한 질문의 응답 예시입니다.



**볼 곳**　‘질문별 결과’ 안에서 ‘첫 번째 후보’를 거쳐 어느 값이 오른쪽에 남는지 보세요.

**애니메이션의 의미**　질문별 검색 결과 안에서 첫 문서의 ID를 꺼냅니다.


### 셀 01 · 바깥 목록과 안쪽 목록

첫 질문에 딸린 ID 목록에서 첫 번째 항목을 골라 보세요.

In [1]:
raw = {"ids": [["library-1"]], "documents": [["도서 대출 기간은 14일입니다."]],
       "metadatas": [[{"source": "가상 도서관 이용 안내", "page": 1}]]}
print(raw["ids"][0][0])

library-1


library-1이 나옵니다. [0]을 한 번만 쓰면 문자열이 아니라 첫 질문의 ID 목록이 남습니다.

### 과제 02 · 출처가 있는 한 건 조립하기

세 열의 같은 위치를 사용해 id, text, source, page가 있는 사전을 만드세요.

In [2]:
# ID와 본문, 메타데이터를 한 건으로 합치세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>meta = raw[&quot;metadatas&quot;][0][0]
row = {&quot;id&quot;: raw[&quot;ids&quot;][0][0], &quot;text&quot;: raw[&quot;documents&quot;][0][0],
       &quot;source&quot;: meta[&quot;source&quot;], &quot;page&quot;: meta[&quot;page&quot;]}
print(row)</code></pre>
<p>실행 결과</p><pre><code>{&#x27;id&#x27;: &#x27;library-1&#x27;, &#x27;text&#x27;: &#x27;도서 대출 기간은 14일입니다.&#x27;, &#x27;source&#x27;: &#x27;가상 도서관 이용 안내&#x27;, &#x27;page&#x27;: 1}</code></pre>
<p>library-1과 1쪽이 같은 사전에 들어갑니다. 여러 결과를 합칠 때는 각 열의 길이가 같은지도 확인합니다. 출처가 누락된 행을 앞 행의 출처로 채우면 안 됩니다.</p>
</details>

## 같은 입력 약속으로 검색 함수 부르기

retrieve(question, k)는 문서 목록을 돌려줍니다. 여기서는 세 키워드의 거리로 계산하는 로컬 검색을 실행합니다. 앞의 Chroma 모양 예시를 실제 조회했다는 뜻은 아닙니다.



**볼 곳**　‘휴관일 질문’에서 출발해 ‘검색 함수’를 거친 뒤 ‘library-2’가 되는 순서를 따라가세요.

**애니메이션의 의미**　질문이 달라도 검색 함수가 반환하는 모양은 같습니다.


### 셀 03 · 휴관 질문에 한 건 요청하기

자료의 원문 ID를 유지합니다. k는 최대 후보 수이고, 질문은 공백을 제외한 문자열이어야 합니다.

휴관 질문에서는 결과 한 건이 있지만 주차 요금에는 대응하는 단어가 없어 빈 목록이 돌아옵니다. 질문을 바꾸면 첫 항목을 읽기 전 목록에 문서가 있는지 확인하세요.

In [3]:
documents = [
  {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1},
  {"id":"library-2","text":"도서관은 월요일에 휴관합니다.","source":"가상 도서관 이용 안내","page":2},
  {"id":"library-3","text":"무인 반납함은 24시간 이용할 수 있습니다.","source":"가상 도서관 이용 안내","page":3}
]

terms = ("대출", "휴관", "반납")

def teaching_vector(text):
    return [float(term in text) for term in terms]

def validate_request(question, k=2):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("질문은 비어 있지 않은 문자열이어야 합니다.")
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("k는 양의 정수여야 합니다.")
    return question.strip()

def retrieve(question, k=2, page=None):
    question = validate_request(question, k)
    query = teaching_vector(question)
    if not any(query):
        return []
    pool = [d for d in documents if page is None or d["page"] == page]
    def rank(doc):
        stored = teaching_vector(doc["text"])
        return (sum((a - b) ** 2 for a, b in zip(query, stored)), doc["id"])
    return sorted(pool, key=rank)[:k]

hits = retrieve("휴관 요일은 언제인가요?", k=1)
if hits:
    print(hits[0]["id"])
else:
    print([])

library-2


library-2는 둘째 쪽의 휴관 안내입니다. k=1은 하나만 가져오라는 입력이지, 그 한 건으로 모든 질문에 답할 수 있다는 보장이 아닙니다.

### 셀 04 · 출처 범위를 1쪽으로 제한하기

휴관 질문은 그대로 두고 검색할 수 있는 쪽만 바꿉니다.

In [4]:
filtered = retrieve("휴관 요일은 언제인가요?", k=1, page=1)
print(filtered[0]["id"], filtered[0]["text"])

library-1 도서 대출 기간은 14일입니다.


1쪽에 남은 library-1은 대출 문장입니다. 진짜 근거가 필터 밖에 있어서 상위 한 건이 질문과 어긋납니다. k만 크게 바꿔도 범위 밖 문서는 돌아오지 않습니다.

## 질문과 근거를 나눠 전달하기

생성 단계에는 사용자가 물은 문장과 검색한 문서를 별개로 전달합니다. 근거에 식별자와 출처를 표시해 두어야 나중에 어느 문장을 썼는지 확인할 수 있습니다.



**볼 곳**　‘사용자 질문’와 ‘출처 붙인 근거’가 오른쪽의 ‘답변 입력’에 함께 들어가는 위치를 보세요.

**애니메이션의 의미**　휴관일 문장에 library-2와 2쪽을 붙여 건넵니다.


### 과제 05 · 검색 문서를 한 줄 제목과 본문으로 표시하기

hits의 첫 문서에서 ID를 꺼내 대괄호로 감싸고 출처와 쪽을 붙인 다음 줄에 본문을 출력하세요.

In [5]:
# hit에서 id, source, page, text를 꺼내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>hit = hits[0]
print(f&quot;[{hit[&#x27;id&#x27;]}] {hit[&#x27;source&#x27;]} / p.{hit[&#x27;page&#x27;]}\n{hit[&#x27;text&#x27;]}&quot;)</code></pre>
<p>실행 결과</p><pre><code>[library-2] 가상 도서관 이용 안내 / p.2
도서관은 월요일에 휴관합니다.</code></pre>
<p>[library-2]와 2쪽이 월요일 휴관 문장 바로 위에 붙습니다. 주차처럼 세 키워드 중 어느 것도 들어 있지 않은 질문에서는 빈 목록이 나오므로 hits[0]을 읽기 전에 목록이 비었는지 확인해야 합니다.</p>
</details>

## 돌아보기

질문을 ‘주차 요금은?’으로 바꿔 retrieve의 반환값을 확인하세요. 빈 목록은 정상 검색 중 근거를 못 찾은 경우입니다. 저장소 연결 실패를 같은 빈 목록으로 바꾸면 다음 장에서 사용자에게 잘못된 이유를 안내하게 됩니다.